# Chapter 11.3. GAE와 PPO가 표준이 된 이유 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter11_3_gae_ppo.ipynb)

책 본문: [Chapter 11.3](https://smhanlab.com/book-ml/kor/ml2/chapter11/3.html)

이 노트북은 PPO를 "실전에 쓸 수 있게" 해준 마지막 두 조각 — **GAE**(TD 오차를
\(\gamma\lambda\)로 지수 가중합한 어드밴티지)와 **엔트로피 보너스** — 를 순서대로
확인합니다: ① 3스텝 손계산(λ가 부호를 뒤집는다) → ② random walk에서 λ의
편향-분산 절충 U자 곡선 → ③ CartPole에서 λ=0.0/0.95/1.0으로 PPO를 학습
비교. 본문의 그림(`ch11_3_gae_lambda_ucurve.svg`, `ch11_3_ppo_lambda_cartpole.svg`,
`ch11_3_ppo_pipeline.svg`)을 이 노트북이 그대로 만들어 냅니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import os
IMG = "/home/smhan/book-ml/kor/src/images"
print("KR font:", kr[0] if kr else "NOT FOUND (확인 필요!)")

KR font: Noto Sans CJK KR


### 워밍업. 3스텝 에피소드: λ만 바꾸면 A₀의 부호가 뒤집힌다

본문과 같은 수치: \(\gamma=0.9\), critic \(V=[2.0,\,1.4,\,0.6,\,0]\)
(\(s_3\) 터미널), 보상 \(r=[1,1,0]\). TD 오차:
\(\delta_0=+0.26,\;\delta_1=+0.14,\;\delta_2=-0.60\).

같은 데이터에서 λ=0이면 가중계수 \((1,0,0)\) → \(A_0=+0.26\)("좋았다"),
λ=1이면 모든 항 포함 → \(A_0=-0.10\)("나빴다") — **신뢰하는 지평선의
길이(λ)에 따라 부호까지** 바뀐다.

In [2]:
gamma = 0.9
V = [2.0, 1.4, 0.6, 0.0]   # s0..s3 (s3 터미널, V=0)
r = [1.0, 1.0, 0.0]
d = [r[t] + gamma * V[t+1] - V[t] for t in range(3)]
print("TD 오차 δ:", [f"{x:+.2f}" for x in d])

for lam in [0.0, 0.5, 0.95, 1.0]:
    w = [(gamma * lam) ** k for k in range(3)]
    A0 = sum(wk * dk for wk, dk in zip(w, d))
    print(f"λ={lam:<4} 가중계수=({w[0]:.4f}, {w[1]:.4f}, {w[2]:.4f})  "
          f"A0_GAE = {A0:+.4f}")

# λ=1 은 원래 정의 G0 - V(s0)와 정확히 같아야 한다
G0 = r[0] + gamma * r[1] + gamma ** 2 * r[2]
assert abs(sum((gamma ** k) * d[k] for k in range(3)) - (G0 - V[0])) < 1e-9
print(f"λ=1 확인: G0 - V(s0) = {G0 - V[0]:+.4f}  (GAE 무한합과 일치 ✓)")

TD 오차 δ: ['+0.26', '+0.14', '-0.60']
λ=0.0  가중계수=(1.0000, 0.0000, 0.0000)  A0_GAE = +0.2600
λ=0.5  가중계수=(1.0000, 0.4500, 0.2025)  A0_GAE = +0.2015
λ=0.95 가중계수=(1.0000, 0.8550, 0.7310)  A0_GAE = -0.0589
λ=1.0  가중계수=(1.0000, 0.9000, 0.8100)  A0_GAE = -0.1000
λ=1 확인: G0 - V(s0) = -0.1000  (GAE 무한합과 일치 ✓)


## 1. GAE 함수: 뒤에서부터 한 번만 스캔

무한합 \(\sum_k (\gamma\lambda)^k \delta_{t+k}\)을 t마다 그대로 합치면
\(O(T^2)\)인데, 재귀

\[A_t^{\text{GAE}} = \delta_t + \gamma\lambda\, A_{t+1}^{\text{GAE}}\]

를 쓰면 **뒤에서 앞에서 한 번만** 스캔하면 된다. 아래 코드는 본문(11.2절
`compute_gae`)과 동일하다.

**자주 하는 실수 두 가지**(본문 11.3):

1. **δ의 부호** — \(r_t + \gamma V(s_t) - V(s_{t+1})\)로 좌변·우변을
   뒤집어 쓰면 모든 어드밴티지가 반전되어 정책이 **역으로** 학습한다.
   "실제로 벌어진 것(\(r_t+\gamma V(s_{t+1})\))"에서 "예측한 것
   (\(V(s_t)\))"을 뺀 것.
2. **last value** — 에피소드가 끝난 뒤의 값. 에피소드가 끝났다면(터미널)
   **0**, 버퍼가 가득 찼는데 에피소드가 안 끝났다면 마지막 상태의
   **\(V(s_T)\)**. 5절의 `collect` 함수가 이 둘을 구분한다.

In [3]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95, last_value=0.0):
    """뒤에서 앞으로 한 번 스캔. values는 rewards보다 1개 길다 (마지막 값 포함)."""
    T = len(rewards)
    gae, advantages = 0.0, [0.0] * T
    for t in reversed(range(T)):
        next_non_terminal = 1.0 if t < T - 1 else 0.0   # 터미널이면 0
        delta = rewards[t] + gamma * values[t+1] * next_non_terminal \
                - values[t]                             # δ_t
        gae = delta + gamma * lam * next_non_terminal * gae   # 재귀
        advantages[t] = gae
    return advantages

# 7-상태 random walk (7.2절과 같은 환경): 0, 6 터미널; 6 도달 시 +1, 0 도달 시 0
# 참값은 정확히 V(s) = s/6 (예: V(3) = 0.5)
V_true = np.array([s / 6 for s in range(7)])

def random_walk_episode(Vw, rng, start=3):
    """Vw(=critic의 값)로 에피소드 1개를 걷고 (보상, critic 값)을 반환."""
    s = start
    rewards, values = [], [Vw[s]]
    while s not in (0, 6):
        s += 1 if rng.random() < 0.5 else -1
        rewards.append(1.0 if s == 6 else 0.0)
        values.append(Vw[s])
    return rewards, values

rng = np.random.default_rng(0)
rw, vw = random_walk_episode(V_true, rng)
print(f"에피소드 길이 {len(rw)}, 총 보상 {sum(rw)}")
A95 = compute_gae(rw, vw, gamma=0.99, lam=0.95)
A0 = compute_gae(rw, vw, gamma=0.99, lam=0.0)
print("λ=0.0 (TD 기반):", [f"{a:+.3f}" for a in A0])
print("λ=0.95(실전 기본):", [f"{a:+.3f}" for a in A95])

에피소드 길이 9, 총 보상 0.0
λ=0.0 (TD 기반): ['-0.170', '+0.162', '+0.160', '+0.158', '-0.173', '-0.172', '-0.170', '-0.168', '-0.167']
λ=0.95(실전 기본): ['-0.336', '-0.176', '-0.359', '-0.552', '-0.756', '-0.619', '-0.476', '-0.325', '-0.167']


## 2. U자 곡선 실험: λ를 바꿔가며 A₀ 추정 MSE를 측정

본문과 같은 셋업 — 참값이 **정확히 알려진 환경**에 **의도적으로 약간
틀린 critic**을 대고 \(A^{\text{GAE}}\)를 추정한다. 이 random walk에서는
왼쪽/오른쪽이 대칭이라 \(Q^{\pi}(s,a)=V^{\pi}(s)\)이고 **참 어드밴티지가
모든 상태에서 0**이므로, 추정치의 MSE가 곧 "편향² + 분산"이다.

여기서는 \(V = V_{\text{true}} + 0.8\,[0,+1,-1,+1,-1,+1,0]\)
(교대 부호의 오차)를 쓴다. 7.2절의 교훈처럼 **4시드 × 에피소드 500회
평균**으로 U자를 확인한다 — 단일 시드로 λ를 고르면 안 된다.

In [4]:
V_wrong = V_true + 0.8 * np.array([0., 1, -1, 1, -1, 1, 0.])  # 의도적으로 틀린 critic
lams = np.round(np.linspace(0, 1, 11), 2)
SEEDS, N_EP = 4, 500

def a0_mse(Vw, lam, seed=0, n_ep=N_EP, gamma=0.99):
    """시드 seed로 n_ep개 에피소드 걷고, 시작 상태 s=3의 A0_GAE^2 평균을 반환."""
    sse = 0.0
    for e in range(n_ep):
        rng = np.random.default_rng(seed * 100003 + e)
        rw, vw = random_walk_episode(Vw, rng)
        a0 = compute_gae(rw, vw, gamma=gamma, lam=lam)[0]
        sse += a0 * a0
    return sse / n_ep

single = {}
for seed in range(SEEDS):
    single[seed] = [a0_mse(V_wrong, lam, seed=seed) for lam in lams]
mean_curve = np.mean(np.stack(list(single.values())), axis=0)

print("λ      : " + "  ".join(f"{l:5.2f}" for l in lams))
print("MSE(시드별):")
for seed, curve in single.items():
    print(f"  {seed}    : " + "  ".join(f"{m:5.3f}" for m in curve))
print("MSE(평균): " + "  ".join(f"{m:5.3f}" for m in mean_curve))
best = lams[int(np.argmin(mean_curve))]
print(f"→ 4시드 평균 U자의 바닥: λ = {best:.1f}")

λ      :  0.00   0.10   0.20   0.30   0.40   0.50   0.60   0.70   0.80   0.90   1.00
MSE(시드별):
  0    : 2.641  2.199  1.856  1.587  1.374  1.206  1.074  0.974  0.908  0.887  0.980
  1    : 2.569  2.130  1.790  1.523  1.310  1.140  1.004  0.899  0.823  0.792  0.881
  2    : 2.580  2.141  1.801  1.533  1.321  1.152  1.019  0.917  0.848  0.826  0.931
  3    : 2.586  2.143  1.800  1.530  1.316  1.145  1.010  0.906  0.834  0.806  0.896
MSE(평균): 2.594  2.153  1.812  1.543  1.330  1.161  1.027  0.924  0.853  0.828  0.922
→ 4시드 평균 U자의 바닥: λ = 0.9


## 3. U자 곡선 그리기

가는 선 = 개별 시드, 굵은 선 = 4시드 평균. 개별 시드만 봐도 U자 모양은
보이지만 바닥 위치가 0.1~0.2 정도 흔들린다 — 평균을 취해야 바닥이 안정된다.
이 환경에서는 바닥이 **λ≈0.9** 부근에 있고, 실전 기본값 λ=0.95는 그 바로
오른쪽(긴 지평선에 거의 의존하되 가장 먼 꼬리만 얕게 신뢰)이다.
편향-분산 절충이 "이론"이 아니라 **측정 가능한 U자**라는 것을 기억하라.

In [5]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for seed, curve in single.items():
    ax.plot(lams, curve, color="0.75", lw=1.0)
ax.plot(lams, mean_curve, color="#1f77b4", lw=2.5, label="4시드 평균")
i = int(np.argmin(mean_curve))
ax.axvline(lams[i], color="crimson", ls="--", lw=1.2)
ax.annotate(f"바닥 λ={lams[i]:.1f} (이 실험)",
            xy=(lams[i], mean_curve[i]),
            xytext=(lams[i] - 0.52, mean_curve[i] * 1.15), color="crimson")
ax.set_xlabel("GAE의 λ")
ax.set_ylabel("A₀ᵍᵃᵉ 추정 MSE (참 어드밴티지 = 0)")
ax.set_title("GAE λ의 편향-분산 절충 — 7-상태 random walk + 의도적으로 틀린 critic")
ax.legend()
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(IMG + "/ch11_3_gae_lambda_ucurve.svg", bbox_inches="tight")
print("저장:", IMG + "/ch11_3_gae_lambda_ucurve.svg",
      os.path.getsize(IMG + "/ch11_3_gae_lambda_ucurve.svg"), "bytes")

저장: /home/smhan/book-ml/kor/src/images/ch11_3_gae_lambda_ucurve.svg 68066 bytes


/tmp/ipykernel_530153/1690667612.py:15: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_530153/1690667612.py:15: UserWarning: Glyph 7501 (\N{MODIFIER LETTER SMALL G}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_530153/1690667612.py:15: UserWarning: Glyph 7491 (\N{MODIFIER LETTER SMALL A}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_530153/1690667612.py:15: UserWarning: Glyph 7497 (\N{MODIFIER LETTER SMALL E}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_530153/1690667612.py:16: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Noto Sans CJK KR.
  fig.savefig(IMG + "/ch11_3_gae_lambda_ucurve.svg", bbox_inches="tight")
/tmp/ipykernel_530153/1690667612.py:16: UserWarning: Glyph 7501 (\N{MODIFIER LETTER SMALL G}) missing from font(s) Noto Sans CJK KR.
  fig.savefig(IMG + "/ch11_3_gae_lambda_ucurve.svg", bbox_inch

## 4. CartPole에서 PPO (λ = 0.0 / 0.95 / 1.0): 모델과 업데이트

CartPole은 이산 행동(0/1) + 이산 보상(매 스텝 +1, 최대 500)이라 11.2절
Pendulum(연속, 30만 스텝에도 −1092)과 달리 **≈7.7만 스텝(150 반복 × 512
스텝)** 안에도 λ에 따른 차이가 명확히 드러난다. 세 λ 모두 **같은 시드(0),
같은 반복 수**로 학습한다.

스택은 본문 "실전 스택의 전체 그림": 경험 수집 → last value →
`compute_gae`(1절) → 미니배치 단위 정규화 → M=4 epochs × 미니배치 64
경사 상승(클립 ε=0.2, critic \(c_v=0.5\), **엔트로피 보너스** \(c_e=0.01\))
→ \(\theta_{\text{old}}\leftarrow\theta\).

In [6]:
import gymnasium as gym
import torch
import torch.nn as nn

SEED = 0
GAMMA = 0.99
ITERS, N_STEPS, EPOCHS, BSIZE = 150, 512, 4, 64
EPS_CLIP, C_V, C_E, LR = 0.2, 0.5, 0.01, 3e-4

S_hist = []          # collect 간 상태 전달 (터미널 후 리셋한 새 에피소드 첫 상태)
START_STATE = None

class CategoricalPolicy(nn.Module):
    """이산 정책 π(a|s) = softmax(W·φ(s)) — CartPole(행동 0/1).
    10.2절의 torch.distributions.Categorical 샘플링과 같다.
    (본문 11.3: PPO는 이산(Categorical)이든 연속(Gaussian, 11.2절)이든
    목적함수·GAE·엔트로피 항의 수학적 형식은 같다 — 변하는 건
    logπ를 어떤 분포에서 계산하느냐뿐이다.)"""
    def __init__(self, n_actions):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(4, 128), nn.Tanh(),
                                   nn.Linear(128, 128), nn.Tanh())
        self.head = nn.Linear(128, n_actions)
    def dist(self, s):
        return torch.distributions.Categorical(logits=self.head(self.trunk(s)))
    def act(self, s, deterministic=False):
        d = self.dist(s)
        a = d.logits.argmax(-1) if deterministic else d.sample()
        return a, d.log_prob(a)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 128), nn.Tanh(),
                                 nn.Linear(128, 128), nn.Tanh(),
                                 nn.Linear(128, 1))
    def forward(self, s):
        return self.net(s).squeeze(-1)

def collect(env, policy, critic, lam):
    """N_STEPS만큼 경험 수집.
    에피소드가 끝나면(①) 바로 reset해서 새 에피소드의 첫 상태를 다음 스텝에 쓴다 —
    gymnasium에서 터미널 이후 step()은 undefined behavior다.
    last value 규칙(본문의 두 번째 실수): 버퍼 마지막이 터미널이면 0,
    에피소드가 안 끝났으면 마지막 상태의 V(s_T)."""
    obs = S_hist[-1] if S_hist else START_STATE
    env.unwrapped.state = np.asarray(obs, dtype=np.float64)  # env 내부 상태 동기화
    S, A, R, LOGP, V, D = [], [], [], [], [], []
    for _ in range(N_STEPS):
        S.append(obs)   # s_t: 행동을 고른 상태 (다음 상태가 아님!)
        st = torch.tensor(obs, dtype=torch.float32)
        with torch.no_grad():
            a, lp = policy.act(st)
            v = critic(st)
        obs, r, term, trunc, _ = env.step(int(a))
        A.append(int(a)); R.append(float(r))
        LOGP.append(float(lp)); V.append(float(v)); D.append(bool(term or trunc))
        if term or trunc:
            obs, _ = env.reset()   # 에피소드 끝 → 다음 스텝은 새 에피소드의 첫 상태
            S_hist.append(obs)     # 다음 collect가 이 상태로 이어가게 기록
    if D[-1]:
        last_value = 0.0   # 에피소드 끝남(터미널)
    else:
        with torch.no_grad():
            last_value = float(critic(torch.tensor(obs, dtype=torch.float32)))  # V(s_T)
    S_hist.append(obs)   # (터미널이면 리셋 후의) 현재 상태를 다음 collect에 전달
    values = V + [last_value]
    adv = compute_gae(R, values, gamma=GAMMA, lam=lam)
    returns = [a_ + v for a_, v in zip(adv, V)]
    return S, A, R, LOGP, V, np.array(adv), np.array(returns), D

def run_ppo(lam, seed=SEED, iters=ITERS):
    """한 λ로 iters 반복. 반환: (에피소드리턴, 에피소드길이, 평가 [(반복,평균,최고)])"""
    global S_hist, START_STATE
    torch.manual_seed(seed); np.random.seed(seed)
    env = gym.make("CartPole-v1")
    policy, critic = CategoricalPolicy(n_actions=2), Critic()
    # 결정적 초기 상태(종속 시드 고정) — 수집 루프가 시작할 때 복원한다
    _obs, _info = env.reset(seed=seed)
    START_STATE = env.unwrapped.state.copy()
    S_hist.clear()
    opt = torch.optim.Adam(list(policy.parameters()) + list(critic.parameters()), lr=LR)
    ep_rets, ep_lens, evals = [], [], []
    part_r, part_l = 0.0, 0
    for it in range(iters):
        S, A, R, LOGP, V, adv, rets, D = collect(env, policy, critic, lam)
        # 에피소드 경계를 따라 (리턴, 길이) 기록
        for rr, dd in zip(R, D):
            part_r += rr; part_l += 1
            if dd:
                ep_rets.append(part_r); ep_lens.append(part_l)
                part_r, part_l = 0.0, 0
        n = len(R); order = np.arange(n)
        for _ in range(EPOCHS):                      # ⑤ M epochs
            np.random.shuffle(order)
            for mb in range(0, n, BSIZE):            # 미니배치
                b = order[mb:mb + BSIZE]
                ab = (adv[b] - adv[b].mean()) / (adv[b].std() + 1e-8)  # 미니배치 정규화
                sB = torch.tensor(np.array(S, dtype=np.float32)[b])
                aB = torch.tensor(A, dtype=torch.long)[b]
                lpb = torch.tensor(np.array(LOGP, dtype=np.float32)[b])
                rnB = torch.tensor(np.array(rets, dtype=np.float32)[b])
                abn_t = torch.tensor(ab)
                dist = policy.dist(sB)
                ratio = torch.exp(dist.log_prob(aB) - lpb)
                pol = -torch.min(ratio * abn_t,
                                 torch.clamp(ratio, 1 - EPS_CLIP, 1 + EPS_CLIP) * abn_t).mean()
                cri = torch.mean((critic(sB) - rnB) ** 2)
                loss = pol + C_V * cri - C_E * dist.entropy().mean()   # 엔트로피 보너스
                opt.zero_grad(); loss.backward(); opt.step()
        if (it + 1) % 25 == 0:                       # 결정적(평균) 정책으로 20회 평가
            r20 = []
            for e in range(20):
                s, _ = env.reset(seed=seed * 100000 + it * 100 + e)
                tot = 0.0
                for _ in range(500):
                    with torch.no_grad():
                        a, _ = policy.act(torch.tensor(s, dtype=torch.float32),
                                          deterministic=True)
                    s, r, term, trunc, _ = env.step(int(a)); tot += r
                    if term or trunc: break
                r20.append(tot)
            evals.append((it, float(np.mean(r20)), float(max(r20))))
    env.close()
    return ep_rets, ep_lens, evals

## 5. 학습하고 비교

세 λ를 각자 시드 0으로 150 반복 학습(≈7.7만 스텝). 학습 중 수집된
**에피소드 리턴**을 기록하고, 25반복마다 **결정적(평균) 정책**으로
20회 평가한다. 단일 시드 실험이므로 "λ=0.95가 항상 이긴다"는 보장이
아니다 — λ에 따라 학습의 **모양**이 달라지는 절충의 존재를 보는 실험이다.

In [7]:
results = {}
for lam in [0.0, 0.95, 1.0]:
    ep_rets, ep_lens, evals = run_ppo(lam)
    results[lam] = (ep_rets, ep_lens, evals)
    f20 = np.mean(ep_rets[:20]); l20 = np.mean(ep_rets[-20:])
    print(f"λ={lam:<4} 처음20={f20:6.1f}  마지막20={l20:6.1f}  최고={max(ep_rets):5.0f}  "
          f"최종평가 평균/최고={evals[-1][1]:5.0f}/{evals[-1][2]:5.0f}  "
          f"(에피소드 수 {len(ep_rets)})")
print("(시드 0 단일 실행 — 본문과 같은 조건. 학습의 '모양'을 보는 것이지 최적 λ 증명 아님.)")

/home/smhan/miniconda3/envs/bookml/lib/python3.11/site-packages/gymnasium/envs/classic_control/cartpole.py:213: UserWarning: WARN: You are calling 'step()' even though this environment has already returned terminated = True. You should always call 'reset()' once you receive 'terminated = True' -- any further steps are undefined behavior.
  logger.warn(


λ=0.0  처음20=  22.6  마지막20=   9.9  최고=   47  최종평가 평균/최고=    9/   10  (에피소드 수 7741)


λ=0.95 처음20=  22.6  마지막20=  13.5  최고=   85  최종평가 평균/최고=    9/   10  (에피소드 수 5426)


λ=1.0  처음20=  22.6  마지막20=  15.2  최고=  100  최종평가 평균/최고=   10/   10  (에피소드 수 5266)
(시드 0 단일 실행 — 본문과 같은 조건. 학습의 '모양'을 보는 것이지 최적 λ 증명 아님.)


In [8]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
colors = {0.0: "#d62728", 0.95: "#1f77b4", 1.0: "#2ca02c"}
labels = {0.0: "λ=0.0 (TD 기반)", 0.95: "λ=0.95 (실전 기본값)", 1.0: "λ=1.0 (MC 기반)"}
for lam, (ep_rets, ep_lens, evals) in results.items():
    steps = np.cumsum(ep_lens)
    k = 10
    if len(ep_rets) >= k:
        rm = np.convolve(ep_rets, np.ones(k) / k, mode="valid")
        ax.plot(steps[k - 1:], rm, color=colors[lam], lw=1.6, label=labels[lam])
    ex = [it * N_STEPS for it, _, _ in evals]
    ey = [m for _, m, _ in evals]
    ax.scatter(ex, ey, s=28, color=colors[lam], edgecolor="k", zorder=5)
ax.axhline(500, color="0.8", ls=":", lw=1)
ax.set_xlabel("누적 환경 스텝")
ax.set_ylabel("에피소드 리턴")
ax.set_title("CartPole PPO (시드 0) — λ별 학습 곡선. 선: 10에피소드 이동평균, 점: 25반복마다 20회 평가")
ax.set_ylim(0, 520)
ax.legend()
fig.tight_layout()
fig.savefig(IMG + "/ch11_3_ppo_lambda_cartpole.svg", bbox_inches="tight")
print("저장:", IMG + "/ch11_3_ppo_lambda_cartpole.svg",
      os.path.getsize(IMG + "/ch11_3_ppo_lambda_cartpole.svg"), "bytes")

저장: /home/smhan/book-ml/kor/src/images/ch11_3_ppo_lambda_cartpole.svg 252729 bytes


## 6. 실전 스택의 전체 그림 (구조도)

본문 "PPO를 돌린다: 실전 스택의 전체 그림"의 ①~⑥ 흐름을 Graphviz 구조도
로 만든다. `.dot` 소스는 `kor/src/images/`에 같은 이름으로 함께 저장된다.

In [9]:
import subprocess

DOT = """digraph ppo_pipeline {
    rankdir=TB;
    graph [fontname="Noto Sans CJK KR",
           label="PPO 한 반복(1 iteration)의 전체 데이터 흐름 — ①경험수집 → ②last value → ③GAE → ④미니배치 정규화 → ⑤M epochs 경사상승 → ⑥θ_old 갱신 → ①로",
           labelloc=t, labeljust=c, bgcolor="white", pad=0.3];
    node  [fontname="Noto Sans CJK KR", shape=box, style="rounded,filled",
           fillcolor="white", fontcolor="#212529", fontsize=11,
           penwidth=1.3, margin="0.2,0.12"];
    edge  [fontname="Noto Sans CJK KR", fontsize=10, color="#495057", penwidth=1.5];

    N1 [label="① 경험 수집: π_old로 N 스텝(2048)\n저장 (s_t, a_t, r_t, logπ_old(a_t|s_t), V_old(s_t))",
          fillcolor="#e7f1ff"];
    N2 [label="② 마지막 상태의 V 계산\n(last value = 에피소드 끝나면 0, 아니면 V(s_T))",
        fillcolor="#e7f1ff"];
    N3 [label="③ compute_gae: 뒤에서 앞으로 한 번 스캔\nA_t = Σ_k (γλ)^k δ_{t+k}   (γ=0.99, λ=0.95)",
        fillcolor="#e7f1ff"];
    N4 [label="④ A_t 를 미니배치 단위로 정규화\n(mean 0, std 1) — 학습률이 보상 스케일에 무관하게",
        fillcolor="#e7f1ff"];

    subgraph cluster5 {
        label="⑤ M epochs(4) × 미니배치(64)마다 경사 상승";
        labeljust=l; fontsize=11; fontcolor="#343a40";
        color="#adb5bd"; style="rounded,filled"; fillcolor="#f8f9fa";
        node [fillcolor="white"];
        L1 [label="정책 손실\n−min(r_t·A_t, clip(r_t, 1±ε)·A_t)   (ε=0.2)"];
        L2 [label="critic 손실\nMSE(V_θ(s_t), R_t)   (R_t = A_t + V(s_t))"];
        L3 [label="엔트로피 보너스\n+c_e · H[π_θ(·|s_t)]   (c_e ≈ 0.01)"];
        L4 [label="전체 손실 = 정책 + c_v·critic − c_e·엔트로피  (최소화)"];
        L1 -> L4; L2 -> L4; L3 -> L4;
    }

    N6 [label="⑥ θ_old ← θ  (다음 반복의 기준 정책 갱신) → ①로",
        fillcolor="#e7f1ff"];

    N1 -> N2 -> N3 -> N4 -> L1 [label="A_t, R_t"];
    N1 -> L2 [label="s_t, R_t", style=dashed];
    L4 -> N6;
    N6 -> N1 [label="다음 반복"];
}"""

dot_path = IMG + "/ch11_3_ppo_pipeline.dot"
svg_path = IMG + "/ch11_3_ppo_pipeline.svg"
with open(dot_path, "w") as f:
    f.write(DOT)
subprocess.run(["dot", "-Tsvg", dot_path, "-o", svg_path], check=True)
print("저장:", svg_path, os.path.getsize(svg_path), "bytes")

저장: /home/smhan/book-ml/kor/src/images/ch11_3_ppo_pipeline.svg 13220 bytes
